# GLMsingle beta-version QC — decoding comparison

Aggregates the per-subject CSVs written by `run_beta_qc_decoding.py` and asks a
single question: **does GLMsingle's denoising/ridge pipeline (B→C→D) improve
decodability on this dataset?**

- **A** = ONOFF, **B** = +FITHRF, **C** = +GLMDENOISE, **D** = +ridge (fracridge).
  **A is written but excluded from decoding** — ONOFF pools every event into one
  on/off beta per voxel, so there's no per-trial dimension to decode. The real
  comparison is **B → C → D**.
- Decoding target: **stimulus category** (whole-brain LinearSVC, accuracy, chance 0.25),
  same standardized decoder / LOGO-CV on every beta version. Category decoding is a
  **pipeline-validation probe** — a high-SNR signal sensitive enough to rank the beta
  versions. It is *not* a result of interest; the analyses of interest (reward/value)
  live in `run_qvalue_*.py`.

B→D is usually but **not guaranteed** monotonic; a dip is informative (e.g. GLMdenoise
removing condition-correlated variance), not a bug.

In [ ]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Directory holding the per-subject sub-*/sub-*_beta_qc_decoding.csv files
# (the --output-dir passed to run_beta_qc_decoding.py). Edit for your filesystem.
QC_DIR = Path("/mnt/data/learning-habits/bids_dataset/derivatives/glmsingle_qc")

BETA_ORDER = ["A", "B", "C", "D"]
BETA_LABELS = {"A": "A\nONOFF", "B": "B\n+FITHRF",
               "C": "C\n+GLMdenoise", "D": "D\n+ridge"}

In [ ]:
# Load and concatenate all per-subject QC CSVs
csvs = sorted(glob.glob(str(QC_DIR / "sub-*" / "sub-*_beta_qc_decoding.csv")))
assert csvs, f"No beta-QC CSVs found under {QC_DIR}"
df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
df["beta_type"] = pd.Categorical(df["beta_type"], categories=BETA_ORDER, ordered=True)

n_sub = df["subject"].nunique()
print(f"{len(csvs)} subjects, targets: {sorted(df['target'].unique())}")
df.head()

In [ ]:
# Group summary: mean ± sem per beta_type x target, plus mean gain relative to type-B
# (type A is excluded from decoding — see intro cell — so B is the comparison baseline)
summary = (df.groupby(["target", "beta_type"], observed=True)["value"]
             .agg(mean="mean", sem=lambda x: x.std(ddof=1) / np.sqrt(x.count()), n="count")
             .reset_index())

for tgt in summary["target"].unique():
    s = summary[summary.target == tgt].set_index("beta_type")
    base = s.loc["B", "mean"] if "B" in s.index else np.nan
    print(f"\n=== {tgt} ({df[df.target==tgt]['metric'].iloc[0]}) ===")
    for bt in BETA_ORDER:
        if bt in s.index:
            m, se = s.loc[bt, "mean"], s.loc[bt, "sem"]
            print(f"  {bt}: {m:.3f} ± {se:.3f}   (Δ vs B: {m - base:+.3f})")
summary

In [ ]:
# Faint per-subject A→D trajectories + group mean ± sem, with chance marked.
tgt = "category"
sub = df[df.target == tgt]
metric = sub["metric"].iloc[0]
baseline = sub["baseline"].iloc[0]
x = np.arange(len(BETA_ORDER))

fig, ax = plt.subplots(figsize=(5.2, 4.4))

# per-subject paired trajectories
wide = sub.pivot_table(index="subject", columns="beta_type",
                       values="value", observed=True).reindex(columns=BETA_ORDER)
for _, row in wide.iterrows():
    ax.plot(x, row.values, color="0.75", lw=0.8, alpha=0.6, zorder=1)

# group mean ± sem
g = summary[summary.target == tgt].set_index("beta_type").reindex(BETA_ORDER)
ax.errorbar(x, g["mean"].values, yerr=g["sem"].values, color="#c1121f",
            lw=2.2, marker="o", ms=7, capsize=4, zorder=3, label="mean ± sem")

ax.axhline(baseline, ls="--", color="0.4", lw=1,
           label=f"chance ({baseline:g})", zorder=2)
ax.set_xticks(x)
ax.set_xticklabels([BETA_LABELS[b] for b in BETA_ORDER])
ax.set_ylabel(metric)
ax.set_title(f"category decoding (n={wide.shape[0]})")
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("GLMsingle beta-version decoding QC", fontweight="bold")
fig.tight_layout()
plt.show()